<a href="https://colab.research.google.com/github/hmmnyamminji/DL/blob/main/day14_practice1_%EB%B9%84%EC%A0%84_%EB%B0%9C%EC%A0%84%EC%82%AC_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
torch.manual_seed(42)

In [16]:
# 셀 1. skip connection 스킵 커넥션 - '출력 = 변환 + 입력 그대로'
# ResNet 의 아이디어: y = F(x) + x    ← 입력을 '그대로 더해주는' 지름길(skip)이 생김
# 역전파 (체인 룰)에서 '+ x'의 미분은 + 1, F(x) 부분의 기울기가 아무리 작아져도, 상수 1이 항상 더해지므로 기울기가 0으로 사라지지 않는다.
# 층을 아무리 지나도 곱해서 줄어들지 않는 고속도로가 생긴다.

class ResidualBlock(nn.Module):
  def __init__(self):
    super().__init__()
    self.f = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64,64))
  def forward(self, x):
    return torch.relu(self.f(x) + x)

def make_residual(n_blocks): # 60층의 layer층 쌓기
  layers = [ResidualBlock() for _ in range(n_blocks)]
  return nn.Sequential(*layers, nn.Linear(64, 1))

def first_grad_res(net):
  torch.manual_seed(0)
  x = torch.rand(32,64)
  net(x).mean().backward() # 순전파 → 평균 → 역전파 (기울기 계산)
  return net[0].f[0].weight.grad.abs().mean().item() # 첫 block 첫 Linear의 기울기 (.weight.grad) 측정

torch.manual_seed(0)
g_res = first_grad_res(make_residual(25))
print(f"\n같은 50층 깊이, 스킵 커넥션 있음: 첫 층 기울기 {g_res:.2e}")

# 평범한 신경망, 첫 층 기울기: 10층 3.28e-06 vs 50층 1.67e-21
# skip_connection : 같은 50층 깊이 첫 층 기울기 3.64e-03

# 152층 ResNet 을 가능하게 했고, 이후 거의 모든 값은 모델(YOLO 내부, 트랜스포머)에 들어간다.


같은 50층 깊이, 스킵 커넥션 있음: 첫 층 기울기 7.15e-03


In [17]:
#일반 신경망

class ResidualBlock(nn.Module):
  def __init__(self):
    super().__init__()
    self.f = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64,64))
  def forward(self, x):
    return torch.relu(self.f(x)) # ← 일반 신경망

def make_residual(n_blocks):
  layers = [ResidualBlock() for _ in range(n_blocks)]
  return nn.Sequential(*layers, nn.Linear(64, 1))

def first_grad_res(net):
  torch.manual_seed(0)
  x = torch.rand(32,64)
  net(x).mean().backward()
  return net[0].f[0].weight.grad.abs().mean().item()
torch.manual_seed(0)
g_res = first_grad_res(make_residual(25))
print(f"\n같은 50층 깊이, 스킵 커넥션 없음: 첫 층 기울기 {g_res:.2e}")



같은 50층 깊이, 스킵 커넥션 없음: 첫 층 기울기 3.14e-21


In [9]:
model_25_blocks = make_residual(25)
print(model_25_blocks)

Sequential(
  (0): ResidualBlock(
    (f): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (1): ResidualBlock(
    (f): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (2): ResidualBlock(
    (f): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (3): ResidualBlock(
    (f): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (4): ResidualBlock(
    (f): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (5): Residu

In [12]:
# ResNet 구조
from torchvision import models # 사전 정의된 유명 모델 모음
resnet = models.resnet18(weights=None) # resnet18, resnet50 을 백본으로 많이 사용한다.
n_params = sum(p.numel() for p in resnet.parameters()) # 총 파라미터 수
print(f"\nresnet18: 파라미터 {n_params/1e6:.1f}M")
print("첫 블록 구경:")
print(resnet.layer1[0])


resnet18: 파라미터 11.7M
첫 블록 구경:
BasicBlock(
  (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
)
